<a href="https://colab.research.google.com/github/logonia/DAP/blob/main/Comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# capfg_complete_comparison.py
# - Trains Baseline and Full CapFG (single run)
# - Generates side-by-side confusion matrices
# - Plots training loss & validation accuracy curves
# - Shows original test images vs. reconstructions (Baseline & Full)
# ============================================================

import os
import random
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam, lr_scheduler
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms

# ---------------------------
# Reproducibility
# ---------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ---------------------------
# Helper functions
# ---------------------------
def squash(inputs, axis=-1):
    norm = torch.norm(inputs, p=2, dim=axis, keepdim=True)
    scale = (norm ** 2) / (1.0 + norm ** 2)
    return scale * inputs / (norm + 1e-8)

def to_onehot(y, num_classes):
    return torch.eye(num_classes, device=y.device)[y]

def caps_loss(y_true, y_pred, x, x_recon, lam_recon):
    margin = (y_true * torch.clamp(0.9 - y_pred, min=0.0)**2 +
              0.5 * (1.0 - y_true) * torch.clamp(y_pred - 0.1, min=0.0)**2)
    margin_loss = margin.sum(dim=1).mean()
    recon_loss = F.mse_loss(x_recon, x)
    return margin_loss + lam_recon * recon_loss

# ---------------------------
# Data loader (CIFAR-10)
# ---------------------------
def load_cifar10(data_dir="./data", batch_size=128, val_size=5000):
    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
    ])
    eval_tf = transforms.Compose([transforms.ToTensor()])

    full_train_aug = datasets.CIFAR10(data_dir, train=True, download=True, transform=train_tf)
    full_train_eval = datasets.CIFAR10(data_dir, train=True, download=False, transform=eval_tf)
    test_set = datasets.CIFAR10(data_dir, train=False, download=True, transform=eval_tf)

    total_len = len(full_train_aug)
    train_len = total_len - val_size
    train_subset, val_subset = random_split(range(total_len), [train_len, val_size],
                                            generator=torch.Generator().manual_seed(42))
    train_set = Subset(full_train_aug, train_subset.indices)
    val_set = Subset(full_train_eval, val_subset.indices)

    pin = torch.cuda.is_available()
    train_loader = DataLoader(train_set, batch_size, shuffle=True, num_workers=2, pin_memory=pin)
    val_loader   = DataLoader(val_set, batch_size, shuffle=False, num_workers=2, pin_memory=pin)
    test_loader  = DataLoader(test_set, batch_size, shuffle=False, num_workers=2, pin_memory=pin)
    return train_loader, val_loader, test_loader

# ---------------------------
# Capsule components (exactly as in your final script)
# ---------------------------
class PrimaryCapsuleBase(nn.Module):
    def __init__(self, in_c=256, maps=32, dims=8):
        super().__init__()
        self.maps, self.dims = maps, dims
        self.conv = nn.Conv2d(in_c, maps*dims, 9, stride=2)
    def forward(self, x):
        out = self.conv(x)
        b, _, h, w = out.shape
        out = out.view(b, self.maps, self.dims, h, w).permute(0,1,3,4,2).contiguous()
        out = squash(out, axis=-1)
        return out.reshape(b, -1, self.dims), h, w

class PrimaryCapsuleUnsquashed(nn.Module):
    def __init__(self, in_c=256, maps=32, dims=8):
        super().__init__()
        self.maps, self.dims = maps, dims
        self.conv = nn.Conv2d(in_c, maps*dims, 9, stride=2)
    def forward(self, x):
        out = self.conv(x)
        b, _, h, w = out.shape
        out = out.view(b, self.maps, self.dims, h, w).permute(0,1,3,4,2).contiguous()
        raw_norms = torch.norm(out, dim=-1)
        return out, raw_norms, h, w

class DenseCapsule(nn.Module):
    def __init__(self, in_caps, out_caps, in_dims, out_dims, routings=3):
        super().__init__()
        self.in_caps = in_caps
        self.out_caps = out_caps
        self.routings = routings
        self.W = nn.Parameter(0.01 * torch.randn(out_caps, in_caps, out_dims, in_dims))
    def forward(self, x):
        u_hat = torch.einsum('bid,oijd->boij', x, self.W)
        b = torch.zeros(x.size(0), self.out_caps, self.in_caps, device=x.device)
        for i in range(self.routings):
            c = F.softmax(b, dim=1)
            s = (c.unsqueeze(-1) * u_hat).sum(dim=2)
            v = squash(s, axis=-1)
            if i < self.routings-1:
                b = b + (u_hat * v.unsqueeze(2)).sum(dim=-1)
        return v

class ImprovedCapFG(nn.Module):
    def __init__(self, in_maps=32, beta=0.2, entropy_weight=0.01, use_global=True):
        super().__init__()
        self.beta = beta
        self.entropy_weight = entropy_weight
        self.use_global = use_global
        self.local = nn.Sequential(
            nn.Conv2d(in_maps, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(inplace=True),
            nn.Conv2d(16, 1, 1)
        )
        if use_global:
            self.global_pool = nn.AdaptiveAvgPool2d(1)
            self.fc = nn.Linear(in_maps, 1)
        self.temperature = nn.Parameter(torch.tensor(1.0))
    def forward(self, raw_norms):
        logits = self.local(raw_norms)
        if self.use_global:
            g = torch.sigmoid(self.fc(self.global_pool(raw_norms).flatten(1)))
            logits = logits * g[:, :, None, None]
        return torch.sigmoid(logits / self.temperature)
    def entropy_loss(self, mask):
        m = mask.view(mask.size(0), -1)
        entropy = -torch.mean(m * torch.log(m+1e-8) + (1-m)*torch.log(1-m+1e-8))
        return self.entropy_weight * entropy

class CapsuleNetAblation(nn.Module):
    def __init__(self, shape=(3,32,32), classes=10, routings=3,
                 use_input_mask=False, use_capfg=False, mask_threshold=0.1,
                 beta=0.2, entropy_weight=0.01, use_global=True):
        super().__init__()
        self.shape = shape
        self.classes = classes
        self.use_input_mask = use_input_mask
        self.use_capfg = use_capfg
        self.mask_threshold = mask_threshold

        self.conv1 = nn.Conv2d(shape[0], 256, kernel_size=9, stride=1, padding=0)
        self.relu = nn.ReLU(inplace=True)

        if use_capfg:
            self.primary = PrimaryCapsuleUnsquashed(256, 32, 8)
            self.mask_gen = ImprovedCapFG(32, beta=beta, entropy_weight=entropy_weight, use_global=use_global)
            self.num_caps_in = 32 * 8 * 8
        else:
            self.primary = PrimaryCapsuleBase(256, 32, 8)
            self.num_caps_in = 32 * 8 * 8

        self.digitcaps = DenseCapsule(self.num_caps_in, classes, 8, 16, routings)
        self.decoder = nn.Sequential(
            nn.Linear(16*classes, 512), nn.ReLU(inplace=True),
            nn.Linear(512, 1024), nn.ReLU(inplace=True),
            nn.Linear(1024, shape[0]*shape[1]*shape[2]), nn.Sigmoid()
        )

    def forward(self, x, y=None, return_mask=False):
        if self.use_input_mask:
            x = x * (x > self.mask_threshold).float()
        out = self.relu(self.conv1(x))
        if self.use_capfg:
            prim_unsq, raw_norms, h, w = self.primary(out)
            mask = self.mask_gen(raw_norms)
            mask_exp = mask.unsqueeze(-1)
            prim_masked = prim_unsq * mask_exp + prim_unsq * self.mask_gen.beta * (1 - mask_exp)
            prim_squashed = squash(prim_masked, axis=-1)
            prim_flat = prim_squashed.view(x.size(0), -1, 8)
        else:
            prim_flat, h, w = self.primary(out)
        out = self.digitcaps(prim_flat)
        length = out.norm(dim=-1)
        if y is None:
            idx = length.max(1)[1]
            y = torch.eye(self.classes, device=x.device)[idx]
        recon = self.decoder((out * y[:,:,None]).view(out.size(0), -1))
        recon = recon.view(-1, *self.shape)
        if return_mask and self.use_capfg:
            return length, recon, mask
        return length, recon

    def mask_entropy_loss(self, mask):
        if self.use_capfg:
            return self.mask_gen.entropy_loss(mask)
        return torch.tensor(0.0, device=device)

# ---------------------------
# Evaluation and training
# ---------------------------
def evaluate_capsule(model, loader, lam_recon, num_classes, use_mask_loss=False):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            y_onehot = to_onehot(y, num_classes)
            if use_mask_loss and hasattr(model, 'mask_entropy_loss'):
                y_pred, x_recon, mask = model(x, return_mask=True)
                loss = caps_loss(y_onehot, y_pred, x, x_recon, lam_recon) + model.mask_entropy_loss(mask)
            else:
                y_pred, x_recon = model(x)
                loss = caps_loss(y_onehot, y_pred, x, x_recon, lam_recon)
            total_loss += loss.item() * x.size(0)
            correct += (y_pred.argmax(1) == y).sum().item()
            total += x.size(0)
    return total_loss/total, correct/total

def train_one_run(model, train_loader, val_loader, config, use_mask_loss, history=None):
    opt = Adam(model.parameters(), lr=config['lr'])
    sched = lr_scheduler.ExponentialLR(opt, gamma=config['lr_decay'])
    best_acc = 0.0
    train_losses, val_accs = [], []
    for epoch in range(config['epochs']):
        model.train()
        epoch_loss = 0.0
        total = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            y_onehot = to_onehot(y, config['classes'])
            opt.zero_grad()
            if use_mask_loss and hasattr(model, 'mask_entropy_loss'):
                y_pred, x_recon, mask = model(x, y_onehot, return_mask=True)
                loss = caps_loss(y_onehot, y_pred, x, x_recon, config['lam_recon']) + model.mask_entropy_loss(mask)
            else:
                y_pred, x_recon = model(x, y_onehot)
                loss = caps_loss(y_onehot, y_pred, x, x_recon, config['lam_recon'])
            loss.backward()
            opt.step()
            epoch_loss += loss.item() * x.size(0)
            total += x.size(0)
        sched.step()
        _, val_acc = evaluate_capsule(model, val_loader, config['lam_recon'], config['classes'], use_mask_loss)
        train_losses.append(epoch_loss/total)
        val_accs.append(val_acc)
        if val_acc > best_acc:
            best_acc = val_acc
        print(f"  Epoch {epoch+1:02d}/{config['epochs']} | loss={epoch_loss/total:.4f} | val_acc={val_acc:.4f} | best={best_acc:.4f}")
    if history is not None:
        history['train_loss'] = train_losses
        history['val_acc'] = val_accs
    return best_acc

def compute_confusion_matrix(model, test_loader, use_mask_loss=False):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            if use_mask_loss and hasattr(model, 'mask_entropy_loss'):
                y_pred, _, _ = model(x, return_mask=True)
            else:
                y_pred, _ = model(x)
            preds = y_pred.argmax(1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    return confusion_matrix(all_labels, all_preds)

def plot_reconstructions(baseline_model, full_model, test_loader, num_samples=5, save_path='reconstruction_comparison.png'):
    baseline_model.eval()
    full_model.eval()
    data_iter = iter(test_loader)
    images, labels = next(data_iter)
    images = images[:num_samples].to(device)
    labels = labels[:num_samples]

    with torch.no_grad():
        _, base_recon = baseline_model(images)
        _, full_recon, _ = full_model(images, return_mask=True)

    # Move to CPU and convert to numpy for plotting
    images_np = images.cpu().numpy().transpose(0, 2, 3, 1)
    base_recon_np = base_recon.cpu().numpy().transpose(0, 2, 3, 1)
    full_recon_np = full_recon.cpu().numpy().transpose(0, 2, 3, 1)

    fig, axes = plt.subplots(num_samples, 3, figsize=(9, 3 * num_samples))
    for i in range(num_samples):
        axes[i, 0].imshow(images_np[i])
        axes[i, 0].set_title(f'Original (label {labels[i].item()})')
        axes[i, 0].axis('off')
        axes[i, 1].imshow(base_recon_np[i])
        axes[i, 1].set_title('Baseline Recon')
        axes[i, 1].axis('off')
        axes[i, 2].imshow(full_recon_np[i])
        axes[i, 2].set_title('Full CapFG Recon')
        axes[i, 2].axis('off')

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"Saved reconstruction comparison to {save_path}")

# ---------------------------
# Main
# ---------------------------
def main():
    config = {
        'epochs': 30,
        'batch_size': 128,
        'lr': 0.001,
        'lr_decay': 0.9,
        'lam_recon': 0.0005 * 3 * 32 * 32,
        'classes': 10,
    }

    print("Loading CIFAR-10...")
    train_loader, val_loader, test_loader = load_cifar10(batch_size=config['batch_size'])

    # Baseline
    print("\n--- Training Baseline CapsNet ---")
    baseline_model = CapsuleNetAblation(shape=(3,32,32), classes=10,
                                        use_input_mask=False, use_capfg=False).to(device)
    baseline_history = {}
    train_one_run(baseline_model, train_loader, val_loader, config,
                  use_mask_loss=False, history=baseline_history)

    # Full CapFG
    print("\n--- Training Full CapFG ---")
    full_model = CapsuleNetAblation(shape=(3,32,32), classes=10,
                                    use_input_mask=True, use_capfg=True, mask_threshold=0.1,
                                    beta=0.2, entropy_weight=0.01, use_global=True).to(device)
    full_history = {}
    train_one_run(full_model, train_loader, val_loader, config,
                  use_mask_loss=True, history=full_history)

    # 1. Confusion matrices
    cm_base = compute_confusion_matrix(baseline_model, test_loader, use_mask_loss=False)
    cm_full = compute_confusion_matrix(full_model, test_loader, use_mask_loss=True)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    ConfusionMatrixDisplay(cm_base, display_labels=range(10)).plot(ax=axes[0], cmap='Blues', values_format='d')
    axes[0].set_title('Baseline CapsNet')
    ConfusionMatrixDisplay(cm_full, display_labels=range(10)).plot(ax=axes[1], cmap='Blues', values_format='d')
    axes[1].set_title('Full CapFG')
    plt.tight_layout()
    plt.savefig('confusion_matrices_comparison.png', dpi=150)
    plt.close()
    print("Saved confusion_matrices_comparison.png")

    # 2. Training curves
    epochs = range(1, config['epochs'] + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(epochs, baseline_history['train_loss'], 'b-', label='Baseline')
    ax1.plot(epochs, full_history['train_loss'], 'r-', label='Full CapFG')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Training Loss')
    ax1.set_title('Training Loss Comparison')
    ax1.legend()
    ax1.grid(True)

    ax2.plot(epochs, baseline_history['val_acc'], 'b-', label='Baseline')
    ax2.plot(epochs, full_history['val_acc'], 'r-', label='Full CapFG')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Validation Accuracy')
    ax2.set_title('Validation Accuracy Comparison')
    ax2.legend()
    ax2.grid(True)
    plt.tight_layout()
    plt.savefig('training_curves_comparison.png', dpi=150)
    plt.close()
    print("Saved training_curves_comparison.png")

    # 3. Reconstruction comparison
    plot_reconstructions(baseline_model, full_model, test_loader, num_samples=5)

    print("\n✅ All plots generated successfully.")

if __name__ == '__main__':
    main()

Using device: cuda
Loading CIFAR-10...


100%|██████████| 170M/170M [00:06<00:00, 24.6MB/s]



--- Training Baseline CapsNet ---
  Epoch 01/30 | loss=0.5718 | val_acc=0.4038 | best=0.4038
  Epoch 02/30 | loss=0.4813 | val_acc=0.4586 | best=0.4586
  Epoch 03/30 | loss=0.4473 | val_acc=0.4904 | best=0.4904
  Epoch 04/30 | loss=0.4227 | val_acc=0.5144 | best=0.5144
  Epoch 05/30 | loss=0.4024 | val_acc=0.5434 | best=0.5434
  Epoch 06/30 | loss=0.3880 | val_acc=0.5260 | best=0.5434
  Epoch 07/30 | loss=0.3760 | val_acc=0.5700 | best=0.5700
  Epoch 08/30 | loss=0.3663 | val_acc=0.5770 | best=0.5770
  Epoch 09/30 | loss=0.3573 | val_acc=0.6036 | best=0.6036
  Epoch 10/30 | loss=0.3496 | val_acc=0.6146 | best=0.6146
  Epoch 11/30 | loss=0.3435 | val_acc=0.6172 | best=0.6172
  Epoch 12/30 | loss=0.3385 | val_acc=0.6360 | best=0.6360
  Epoch 13/30 | loss=0.3311 | val_acc=0.6356 | best=0.6360
  Epoch 14/30 | loss=0.3264 | val_acc=0.6502 | best=0.6502
  Epoch 15/30 | loss=0.3217 | val_acc=0.6556 | best=0.6556
  Epoch 16/30 | loss=0.3180 | val_acc=0.6482 | best=0.6556
  Epoch 17/30 | loss=